# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python)

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [1]:
# importar librerías
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sn
import numpy as np
import pathlib as Path



In [2]:
# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [3]:
# explorar datasets
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


In [4]:
catalog.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 356.0+ bytes


In [5]:
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas

---

In [6]:
#Convertir Fechas en datasets orders y marketing
orders["fecha_hora_pedido"] = pd.to_datetime(orders["fecha_hora_pedido"])
marketing["fecha"] = pd.to_datetime(marketing["fecha"])


# Validar registros en la tabla orders
print("minimos en Orders:")
print(orders[["cantidad", "precio_unitario", "monto_descuento", "monto_total"]].min())

#Calcular monto Teórico esperado
monto_calculado = (orders["cantidad"] * orders["precio_unitario"] - orders["monto_descuento"])

#Detectar inconsistencias (Tolerando centavos por redondeos usando un margen de 0.1)
inconsistencias = orders[abs(orders["monto_total"] - monto_calculado) > 0.1]

print(f"Cantidad de filas con montos inconsistentes: {len(inconsistencias)}")


minimos en Orders:
cantidad            -2.00
precio_unitario     20.03
monto_descuento      0.00
monto_total       -492.65
dtype: float64
Cantidad de filas con montos inconsistentes: 3


In [7]:
inconsistencias.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
266,order_266,user_7011,2025-03-13,NaN,desktop,paid_search,Phone-Pro-128GB,Electronica,-2.0,101.31,10.0,-192.62
267,order_267,user_1087,2025-05-07,NaN,desktop,social,Phone-Pro-128GB,Electronica,-1.0,43.50,5.0,-38.50
268,order_268,user_84,2025-02-19,NaN,desktop,organic,Phone-Pro-128GB,Electronica,-1.0,497.65,5.0,-492.65


In [8]:
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   fecha       1620 non-null   datetime64[ns]
 1   pais        1620 non-null   object        
 2   id_campaña  1620 non-null   object        
 3   canal       1519 non-null   object        
 4   gasto       1620 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 63.4+ KB


In [9]:
# Revisar y eliminar en orders
duplicados_orders = orders.duplicated().sum()
if duplicados_orders > 0:
    orders = orders.drop_duplicates()
    print(f"Se eliminaron {duplicados_orders} registros duplicados en orders.")

# Revisar y eliminar en marketing
duplicados_mkt = marketing.duplicated().sum()
if duplicados_mkt > 0:
    marketing = marketing.drop_duplicates()
    print(f"Se eliminaron {duplicados_mkt} registros duplicados en marketing.")

Se eliminaron 100 registros duplicados en orders.


In [10]:
# Revisar categorías en orders
print("Países únicos en orders:", orders['pais'].unique())
print("Dispositivos únicos:", orders['dispositivo'].unique())
print("Canales de referencia:", orders['fuente_referencia'].unique())

# Estandarizar textos
for col in ['pais', 'dispositivo', 'fuente_referencia', 'categoria_producto']:
    orders[col] = orders[col].astype(str).str.strip().str.lower()

marketing['pais'] = marketing['pais'].astype(str).str.strip().str.lower()
marketing['canal'] = marketing['canal'].astype(str).str.strip().str.lower()

Países únicos en orders: ['Argentina' 'Mexico' 'Colombia' 'mexico' 'colombia' 'argentina' nan]
Dispositivos únicos: ['desktop' 'mobile' nan]
Canales de referencia: ['organic' 'paid_search' 'social' nan]


In [11]:
# 1. Limpieza y estandarización de la columna 'pais'
orders['pais'] = orders['pais'].astype(str).str.strip().str.lower()
# Cambiar el texto 'nan' por 'no especificado'
orders['pais'] = orders['pais'].replace('nan', 'no especificado')
# Capitalizar la primera letra para que se vea ejecutivo ('mexico' -> 'Mexico')
orders['pais'] = orders['pais'].str.capitalize()

# 2. Limpieza de 'dispositivo'
orders['dispositivo'] = orders['dispositivo'].astype(str).str.strip().str.lower()
orders['dispositivo'] = orders['dispositivo'].replace('nan', 'no especificado')

# 3. Limpieza de 'fuente_referencia'
orders['fuente_referencia'] = orders['fuente_referencia'].astype(str).str.strip().str.lower()
orders['fuente_referencia'] = orders['fuente_referencia'].replace('nan', 'no especificado')


# VERIFICACIÓN DE RESULTADOS

print("Países únicos corregidos:", orders['pais'].unique())
print("Dispositivos únicos corregidos:", orders['dispositivo'].unique())
print("Canales de referencia corregidos:", orders['fuente_referencia'].unique())

Países únicos corregidos: ['Argentina' 'Mexico' 'Colombia' 'No especificado']
Dispositivos únicos corregidos: ['desktop' 'mobile' 'no especificado']
Canales de referencia corregidos: ['organic' 'paid_search' 'social' 'no especificado']


In [12]:
# Estandarizar marketing
marketing['pais'] = marketing['pais'].astype(str).str.strip().str.lower().str.capitalize()
marketing['pais'] = marketing['pais'].replace('nan', 'no especificado')

marketing['canal'] = marketing['canal'].astype(str).str.strip().str.lower()
marketing['canal'] = marketing['canal'].replace('nan', 'no especificado')

print("Países únicos corregidos:", marketing['pais'].unique())
print("Canales únicos corregidos:", marketing['canal'].unique())

Países únicos corregidos: ['Mexico' 'Colombia' 'Argentina']
Canales únicos corregidos: ['organic' 'paid_search' 'social' 'no especificado']


In [13]:
# Se crea DataFrame exclusivo para analizar las cancelaciones/devoluciones después
devoluciones = orders[orders['cantidad'] < 0].copy()

# Mantener en 'orders' únicamente las ventas exitosas y confirmadas
orders = orders[orders['cantidad'] > 0].copy()

# 3. Validar que ya no queden montos negativos en la tabla principal de ventas
print(f"Órdenes de venta activas: {orders.shape[0]}")
print(f"Órdenes en proceso de devolución/cancelación aisladas: {devoluciones.shape[0]}")
print(f"Mínimo monto total en ventas: {orders['monto_total'].min()}")

Órdenes de venta activas: 24946
Órdenes en proceso de devolución/cancelación aisladas: 4
Mínimo monto total en ventas: 5.24


In [14]:
orders.info()
print("-------------------------")
catalog.info()
print("-------------------------")
marketing.info()

<class 'pandas.core.frame.DataFrame'>
Index: 24946 entries, 0 to 24999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           24946 non-null  object        
 1   id_usuario          24946 non-null  object        
 2   fecha_hora_pedido   24946 non-null  datetime64[ns]
 3   pais                24946 non-null  object        
 4   dispositivo         24946 non-null  object        
 5   fuente_referencia   24946 non-null  object        
 6   nombre_producto     24916 non-null  object        
 7   categoria_producto  24946 non-null  object        
 8   cantidad            24946 non-null  float64       
 9   precio_unitario     24946 non-null  float64       
 10  monto_descuento     24946 non-null  float64       
 11  monto_total         24946 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(7)
memory usage: 2.5+ MB
-------------------------
<class 'pandas.

In [15]:
# Código para eliminar las 30 filas donde el nombre de producto es nulo.
orders = orders.dropna(subset=["nombre_producto"])

# Validación rápida.

print(f"Dataset definitivo de ordenes para exportar: {orders.shape[0]} filas.")


Dataset definitivo de ordenes para exportar: 24916 filas.


---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [ ]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)?
- ¿Cuál es el costo total?
- ¿Cuánto se ha invertido en marketing?
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden?
- ¿Cuál es la cantidad promedio de productos por orden?
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal?

In [16]:
#Código para unir órdenes con catálogo para obtener los costos del producto
orders_complete = pd.merge(orders, catalog[["nombre_producto", "costo_unitario"]], on="nombre_producto", how='left')


In [17]:
#1. Ingreso total
revenue_total = orders_complete["monto_total"].sum()
#2. Costo Total de los productos
costo_total_productos = (orders_complete["cantidad"] * orders_complete["costo_unitario"]).sum()

# 3. Inversión Total en Marketing
marketing_total = marketing['gasto'].sum()

# 4. Cálculo del Profit (Ganancia Neta)
profit_total = revenue_total - (costo_total_productos + marketing_total)
margen_neto_porcentaje = (profit_total / revenue_total) * 100


print("=== 📊 PARTE 1: RENTABILIDAD DEL NEGOCIO ===")
print(f"Revenue Total: ${revenue_total:,.2f}")
print(f"Costo de Producto Total: ${costo_total_productos:,.2f}")
print(f"Inversión en Marketing: ${marketing_total:,.2f}")
print(f"Profit Neto: ${profit_total:,.2f}")
print(f"Margen Neto de Ganancia: {margen_neto_porcentaje:.2f}%")
print(f"¿Es rentable?: {'✅ SÍ es rentable' if profit_total > 0 else '❌ NO es rentable'}")



=== 📊 PARTE 1: RENTABILIDAD DEL NEGOCIO ===
Revenue Total: $51,954,718.94
Costo de Producto Total: $43,124,069.01
Inversión en Marketing: $2,871,843.53
Profit Neto: $5,958,806.40
Margen Neto de Ganancia: 11.47%
¿Es rentable?: ✅ SÍ es rentable


In [18]:
# Agrupación por pedido para obtener métricas por transacción única
metricas_por_orden = orders_complete.groupby("id_pedido").agg({
    "monto_total": "sum",
    "cantidad": "sum"
})


#1. Ticket Promedio por orden
ticket_promedio = metricas_por_orden['monto_total'].mean()

# 2. Cantidad Promedio de Productos por Orden
cantidad_promedio_orden = metricas_por_orden['cantidad'].mean()

# 3. Producto más vendido (por volumen total de unidades)
producto_mas_vendido = orders_complete.groupby('nombre_producto')['cantidad'].sum().idxmax()
unidades_producto_mas_vendido = orders_complete.groupby('nombre_producto')['cantidad'].sum().max()

# 4. Gasto en Marketing por Canal
gasto_por_canal = marketing.groupby('canal')['gasto'].sum().sort_values(ascending=False)


# ==========================================
# IMPRESIÓN DE RESULTADOS (PARTE 2)
# ==========================================
print("\n=== 📈 PARTE 2: COMPORTAMIENTO DE VENTAS ===")
print(f"🛒 Ticket Promedio por Orden: ${ticket_promedio:,.2f}")
print(f"🛍️ Cantidad Promedio de Productos por Orden: {cantidad_promedio_orden:.2f} unidades")
print(f"🔥 Producto más vendido: '{producto_mas_vendido}' ({unidades_producto_mas_vendido:,.0f} unidades)")
print("\n📣 Gasto en Marketing por Canal:")
print(gasto_por_canal.map('${:,.2f}'.format))






=== 📈 PARTE 2: COMPORTAMIENTO DE VENTAS ===
🛒 Ticket Promedio por Orden: $2,085.20
🛍️ Cantidad Promedio de Productos por Orden: 7.12 unidades
🔥 Producto más vendido: 'Laptop-Gaming-16GB' (144,198 unidades)

📣 Gasto en Marketing por Canal:
canal
social             $918,043.21
organic            $913,533.01
paid_search        $863,088.21
no especificado    $177,179.10
Name: gasto, dtype: object


---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [19]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [20]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [21]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios_unicos,
    CASE
        WHEN nombre_evento = 'first_visit' THEN 1
        WHEN nombre_evento = 'select_item' THEN 2
        WHEN nombre_evento = 'add_to_cart' THEN 3
        WHEN nombre_evento = 'begin_checkout' THEN 4
        WHEN nombre_evento = 'add_payment_info' THEN 5
        WHEN nombre_evento = 'purchase' THEN 6
        ELSE 7
    END AS paso_funnel
FROM events
GROUP BY nombre_evento
ORDER BY paso_funnel ASC;
'''

totals = pd.read_sql(query_totals, con=engine)
totals




,nombre_evento,usuarios_unicos,paso_funnel
0,first_visit,7796,1
1,select_item,7582,2
2,add_to_cart,7634,3
3,begin_checkout,7208,4
4,add_payment_info,6250,5
5,purchase,6240,6


In [22]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH funnel_totals AS (
    SELECT
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios_unicos,
        CASE
            WHEN nombre_evento = 'first_visit' THEN 1
            WHEN nombre_evento = 'select_item' THEN 2
            WHEN nombre_evento = 'add_to_cart' THEN 3
            WHEN nombre_evento = 'begin_checkout' THEN 4
            WHEN nombre_evento = 'add_payment_info' THEN 5
            WHEN nombre_evento = 'purchase' THEN 6
            ELSE 7
        END AS paso_funnel
    FROM events
    GROUP BY nombre_evento
),
funnel_with_initial AS (
    SELECT
        paso_funnel,
        nombre_evento,
        usuarios_unicos,
        LAG(usuarios_unicos, 1) OVER (ORDER BY paso_funnel) AS usuarios_paso_anterior,
        MAX(CASE WHEN paso_funnel = 1 THEN usuarios_unicos END) OVER () AS usuarios_iniciales
    FROM funnel_totals
)
SELECT
    paso_funnel,
    nombre_evento,
    usuarios_unicos,
    COALESCE(LAG(usuarios_unicos, 1) OVER (ORDER BY paso_funnel) - usuarios_unicos, 0) AS usuarios_perdidos,
    CASE
        WHEN usuarios_paso_anterior IS NULL THEN 100.0
        ELSE ROUND((usuarios_unicos * 100.0) / usuarios_paso_anterior, 2)
    END AS conversion_vs_anterior_pct,
    ROUND((usuarios_unicos * 100.0) / usuarios_iniciales, 2) AS conversion_global_pct
FROM funnel_with_initial
ORDER BY paso_funnel ASC;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion



,paso_funnel,nombre_evento,usuarios_unicos,usuarios_perdidos,conversion_vs_anterior_pct,conversion_global_pct
0,1,first_visit,7796,0,100.00,100.00
1,2,select_item,7582,214,97.26,97.26
2,3,add_to_cart,7634,-52,100.69,97.92
3,4,begin_checkout,7208,426,94.42,92.46
4,5,add_payment_info,6250,958,86.71,80.17
5,6,purchase,6240,10,99.84,80.04


---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users`
- `user_activity`

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [23]:

# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)


,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [24]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)


,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [25]:
# Para ver qué columnas tiene la tabla de actividad
print(pd.read_sql("SELECT * FROM user_activity LIMIT 5;", con=engine).columns)

Index(['id_usuario', 'fecha_actividad', 'dias_despues_registro', 'activo'], dtype='object')


In [26]:
# Para ver una muestra de los datos y el "indicador" que menciona el revisor
pd.read_sql("SELECT * FROM user_activity LIMIT 5;", con=engine)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1
3,user_0,2025-02-26,28,0
4,user_1,2025-01-14,7,0


In [27]:
# Retención por cohortes (VERSIÓN FINAL CORREGIDA CON FILTRO DE ACTIVIDAD)
# =====================================================================

query_cohort_retention_final = '''
WITH user_cohorts AS (
    -- 1. Identificar el mes de registro (cohorte) de cada usuario
    SELECT
        id_usuario,
        CAST(fecha_registro AS DATE) AS fecha_reg,
        TO_CHAR(CAST(fecha_registro AS DATE), 'YYYY-MM') AS cohorte
    FROM users
),
activity_weeks AS (
    -- 2. Filtrar SOLO los registros donde el usuario SÍ estuvo activo (activo = 1)
    SELECT
        c.cohorte,
        c.id_usuario,
        a.dias_despues_registro
    FROM user_cohorts c
    INNER JOIN user_activity a ON c.id_usuario = a.id_usuario
    WHERE a.activo = 1  -- <-- ¡La corrección del revisor! Solo contamos actividad real
),
weeks_assigned AS (
    -- 3. Agrupar en los rangos semanales correspondientes
    SELECT
        cohorte,
        id_usuario,
        CASE
            WHEN dias_despues_registro BETWEEN 1 AND 7 THEN 1
            WHEN dias_despues_registro BETWEEN 8 AND 14 THEN 2
            WHEN dias_despues_registro BETWEEN 15 AND 21 THEN 3
            ELSE 0
        END AS semana_desde_registro
    FROM activity_weeks
),
cohort_sizes AS (
    -- 4. Calcular la base de clientes iniciales por mes
    SELECT
        cohorte,
        COUNT(DISTINCT id_usuario) AS clientes_iniciales
    FROM user_cohorts
    GROUP BY cohorte
),
retention_counts AS (
    -- 5. Contar usuarios únicos retenidos por semana objetivo
    SELECT
        cohorte,
        COUNT(DISTINCT CASE WHEN semana_desde_registro = 1 THEN id_usuario END) AS retenido_w1,
        COUNT(DISTINCT CASE WHEN semana_desde_registro = 2 THEN id_usuario END) AS retenido_w2,
        COUNT(DISTINCT CASE WHEN semana_desde_registro = 3 THEN id_usuario END) AS retenido_w3
    FROM weeks_assigned
    GROUP BY cohorte
)
-- 6. Construir la matriz de retención con porcentajes reales
SELECT
    r.cohorte,
    s.clientes_iniciales,
    CAST(r.retenido_w1 AS INTEGER) AS retenido_w1,
    CAST(r.retenido_w2 AS INTEGER) AS retenido_w2,
    CAST(r.retenido_w3 AS INTEGER) AS retenido_w3,
    ROUND((r.retenido_w1 * 100.0) / s.clientes_iniciales, 2) AS semana_1,
    ROUND((r.retenido_w2 * 100.0) / s.clientes_iniciales, 2) AS semana_2,
    ROUND((r.retenido_w3 * 100.0) / s.clientes_iniciales, 2) AS semana_3
FROM retention_counts r
JOIN cohort_sizes s ON r.cohorte = s.cohorte
ORDER BY r.cohorte ASC;
'''

# Ejecutar la consulta en Jupyter
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte,clientes_iniciales,retenido_w1,retenido_w2,retenido_w3,semana_1,semana_2,semana_3
0,2025-01,1627,697,668,656,42.84,41.06,40.32
1,2025-02,1444,611,609,635,42.31,42.17,43.98
2,2025-03,1636,677,705,690,41.38,43.09,42.18
3,2025-04,1606,680,697,663,42.34,43.40,41.28
4,2025-05,1687,695,676,706,41.20,40.07,41.85


Análisis Ejecutivo de Retención Semanal:

Tras corregir el criterio de conteo y evaluar de forma estricta el indicador de actividad real, se observa que la plataforma RappiPlus presenta un comportamiento de retención sumamente maduro y lineal.

A diferencia de los modelos de negocio digitales tradicionales —donde la retención suele desplomarse drásticamente conforme avanzan las semanas (curva de decaimiento logarítmico)—, nuestra plataforma logra estabilizar su base activa de manera inmediata. Tras el registro inicial, aproximadamente 4 de cada 10 usuarios regresan e interactúan de forma recurrente y orgánica en la semana 1, 2 y 3, sin importar el mes de adquisición (cohorte).

Métricas de Negocio Clave:

Tasa de Retención Promedio: Se consolida de forma robusta en un ~42% a lo largo de todo el ciclo de vida de 21 días estudiado.

Consistencia Inter-cohorte: La desviación entre los diferentes meses de registro es mínima (apenas varían un 1% o 2% entre sí), lo que demuestra que la calidad de los usuarios adquiridos a lo largo del año se mantiene constante y que los cambios iterativos en el onboarding han mantenido un impacto predictivo y controlado.

Recomendación Estratégica: Dado que la retención se estabiliza de forma plana en el 42%, el equipo de Producto no debe preocuparse por pérdidas aceleradas de usuarios durante las primeras tres semanas. La pauta estratégica debe enfocarse en campañas de reactivación enfocadas en el 58% de usuarios que quedan inactivos tras su primer registro.

---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado**
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** La nueva modificación en la UI del checkout no tiene un impacto significativo en la tasa de conversión de compra. La conversión del grupo tratamiento es estadísticamente igual a la del grupo control.
   - **H₁ (Hipótesis alternativa):** La nueva modificación en la UI del checkout sí tiene un impacto significativo en la tasa de conversión de compra. La conversión del grupo tratamiento es estadísticamente diferente a la del grupo control.
   
**Test estadístico:** Z-test para dos proporciones independientes.

**Nivel de significancia alpha:** $0.05$ ($5\%$).

In [29]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest

#1. Carga del dataset para realizar el test estádistico
df_exp = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')

# Visualización de control para verificar los nombres en la consola
print("--- Distribución de usuarios por variante ---")
print(df_exp['variante'].value_counts())

# 2. Calcular las conversiones y el total de usuarios por variante
conversion_by_group = df_exp.groupby('variante')['convirtio'].agg(['sum', 'count'])

# Extraer de forma exacta los datos para el Z-Test usando los nombres del dataset
exitos_control = conversion_by_group.loc['control', 'sum']
total_control = conversion_by_group.loc['control', 'count']

exitos_treatment = conversion_by_group.loc['tratamiento', 'sum']
total_treatment = conversion_by_group.loc['tratamiento', 'count']

# Estructurar arreglos para el test estadístico (Tratamiento vs Control)
counts = [exitos_treatment, exitos_control]
nobs = [total_treatment, total_control]

# 3. Aplicar el Z-Test de Proporciones de dos colas
z_stat, p_value = proportions_ztest(counts, nobs, alternative='two-sided')

# 4. Calcular las tasas de conversión descriptivas
tasa_control = (exitos_control / total_control) * 100
tasa_treatment = (exitos_treatment / total_treatment) * 100

print("\n=======================================================")
print("📊 RESULTADOS DEL EXPERIMENTO (A/B TEST)")
print("=======================================================")
print(f"🛒 Tasa de Conversión - Grupo Control: {tasa_control:.2f}%")
print(f"🚀 Tasa de Conversión - Grupo Tratamiento: {tasa_treatment:.2f}%")
print(f"📈 Diferencia absoluta: {tasa_treatment - tasa_control:+.2f}%")
print(f"⏱️ Métrica Z-Statistic: {z_stat:.4f}")
print(f"🔮 P-Value: {p_value:.6f}")
print("-------------------------------------------------------")

# 5. Interpretación del resultado basada en Alpha = 0.05
alpha = 0.05
if p_value < alpha:
    print("✅ RECHAZAMOS LA HIPÓTESIS NULA (H₀).")
    print("El cambio en la UI del checkout TIENE un impacto estadísticamente significativo en la conversión.")
    if tasa_treatment > tasa_control:
        print("👉 Recomendación: Implementar la NUEVA UI al 100% de los usuarios. Genera más valor.")
    else:
        print("👉 Recomendación: Mantener la UI ANTERIOR. El nuevo diseño perjudicó la conversión.")
else:
    print("❌ NO SE RECHAZA LA HIPÓTESIS NULA (H₀).")
    print("No hay evidencia estadística suficiente para afirmar que el cambio en la UI genere un impacto.")
    print("👉 Recomendación: Mantener la versión control, la diferencia observada se debe al azar.")




--- Distribución de usuarios por variante ---
variante
tratamiento    5035
control        4965
Name: count, dtype: int64

📊 RESULTADOS DEL EXPERIMENTO (A/B TEST)
🛒 Tasa de Conversión - Grupo Control: 15.69%
🚀 Tasa de Conversión - Grupo Tratamiento: 16.29%
📈 Diferencia absoluta: +0.60%
⏱️ Métrica Z-Statistic: 0.8133
🔮 P-Value: 0.416059
-------------------------------------------------------
❌ NO SE RECHAZA LA HIPÓTESIS NULA (H₀).
No hay evidencia estadística suficiente para afirmar que el cambio en la UI genere un impacto.
👉 Recomendación: Mantener la versión control, la diferencia observada se debe al azar.


---

---